# Rec04 LFP Download and Processing

This notebook downloads and processes the session 4 raw LFP recording (`rec04.lfp`) independently of the behavior analysis.

In [ ]:
# Configure session paths and parameters
from pathlib import Path
import os
import sys

import numpy as np
import requests
import scipy.io as iomat

PROJECT_DIR = Path.cwd()
HELPER_MODULE = PROJECT_DIR / "get_data.py"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import get_data

LFP_URL = "https://ndownloader.figshare.com/files/43584906"
BEHAV_URL = "https://api.figshare.com/v2/file/download/43845669"
RAW_LFP = PROJECT_DIR / "rec04.lfp"
BEHAV_FILE = PROJECT_DIR / "rec04_BehavElectrDataLFP.mat"
PROCESSED_NPZ = PROJECT_DIR / "rec04_lfps.npz"
PROCESSED_MAT = PROJECT_DIR / "rec04_lfps.mat"

n_channels = 256
init_fs = 1250
fs = 25

print("Project directory:", PROJECT_DIR)
print("Helper module:", HELPER_MODULE)
print("Raw LFP path:", RAW_LFP)
print("Behavior file:", BEHAV_FILE)

Project directory: /home/azosion/Projects/comp-neuro/final code
Helper module: /home/azosion/Projects/comp-neuro/final code/get_data.py
Raw LFP path: /home/azosion/Projects/comp-neuro/final code/rec04.lfp


## Download Raw Session 4 LFP

Downloads `rec04.lfp` from the Figshare direct file URL and verifies that the download completed.

In [5]:
EXPECTED_SIZE = 2347776000

response = requests.get(LFP_URL, timeout=600, stream=True)
response.raise_for_status()

if "text" in response.headers.get("Content-Type", "").lower():
    raise ValueError("The download endpoint returned text instead of binary LFP data.")

with RAW_LFP.open("wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

raw_bytes = RAW_LFP.stat().st_size
if raw_bytes != EXPECTED_SIZE:
    raise ValueError(
        f"Downloaded file size is {raw_bytes} bytes; expected {EXPECTED_SIZE} bytes."
    )

print("Downloaded:", RAW_LFP)
print("File size:", raw_bytes, "bytes")
print("Exists:", RAW_LFP.is_file())

Downloaded: /home/azosion/Projects/comp-neuro/final code/rec04.lfp
File size: 2347776000 bytes
Exists: True


## Set and Validate LFP Metadata

Session 4 uses 256 channels, 1250 Hz acquisition, and 25 Hz decimation.

In [6]:
if not RAW_LFP.is_file():
    raise FileNotFoundError(f"Missing raw LFP file: {RAW_LFP}")

raw_bytes = RAW_LFP.stat().st_size
int16_bytes = np.dtype("int16").itemsize

with RAW_LFP.open("rb") as f:
    signature = f.read(8)

if signature.startswith(b"\x89PNG") or signature.startswith(b"%PDF"):
    raise ValueError("The raw LFP file contains image/PDF data instead of int16 binary samples.")
if raw_bytes < int16_bytes * n_channels:
    raise ValueError("The raw LFP file is too small to contain one full frame for all channels.")
if raw_bytes % int16_bytes != 0:
    raise ValueError("The raw file size is not divisible by two bytes per int16 sample.")

all_int16_samples = raw_bytes // int16_bytes
if all_int16_samples % n_channels != 0:
    raise ValueError("The raw file size is not divisible by the session channel count.")

expected_samples = all_int16_samples // n_channels

print("n_channels:", n_channels)
print("init_fs:", init_fs, "Hz")
print("fs:", fs, "Hz")
print("Raw int16 samples:", all_int16_samples)
print("Expected samples per channel:", expected_samples)
print("Raw duration:", expected_samples / init_fs, "seconds")

n_channels: 256
init_fs: 1250 Hz
fs: 25 Hz
Raw int16 samples: 1173888000
Expected samples per channel: 4585500
Raw duration: 3668.4 seconds


## Process and Inspect the LFP Data

Use `get_data.get_LFP` to decimate the raw recording and verify the resulting array.

In [7]:
lfps = get_data.get_LFP(
    str(RAW_LFP),
    n_channels=n_channels,
    init_fs=init_fs,
    fs=fs,
)

print("LFP array shape:", lfps.shape)
print("Retained channels:", lfps.shape[1])
print("Decimated samples:", lfps.shape[0])
print("Output duration:", lfps.shape[0] / fs, "seconds")
print("Sampling rate:", fs, "Hz")
print("Contains finite values:", bool(np.isfinite(lfps).all()))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
LFP array shape: (91710, 192)
Retained channels: 192
Decimated samples: 91710
Output duration: 3668.4 seconds
Sampling rate: 25 Hz
Contains finite values: True


## Save the Processed LFP Output

Saves the decimated array as both NumPy and MATLAB-compatible files for use in the analysis notebook.

## Process Session 4 Behavioral Data

Generate `lapID` from the session 4 behavior MATLAB file so it can be saved alongside `lfps`.

In [ ]:
if not BEHAV_FILE.is_file():
    behavior_response = requests.get(BEHAV_URL, timeout=600)
    behavior_response.raise_for_status()
    BEHAV_FILE.write_bytes(behavior_response.content)

lapID = get_data.get_behav(str(BEHAV_FILE), fs=25)

print("lapID shape:", lapID.shape)
print("lapID columns:", lapID.shape[1])
print("lapID rows:", lapID.shape[0])

In [ ]:
np.savez(
    PROCESSED_NPZ,
    lfps=lfps,
    lapID=lapID,
    fs=fs,
    init_fs=init_fs,
    n_channels=n_channels,
)
iomat.savemat(
    PROCESSED_MAT,
    {
        "lfps": lfps,
        "lapID": lapID,
        "fs": fs,
        "init_fs": init_fs,
        "n_channels": n_channels,
    },
)

loaded = iomat.loadmat(PROCESSED_MAT)
print("Saved MATLAB file:", PROCESSED_MAT)
print("MATLAB lapID shape:", loaded["lapID"].shape)
print("MATLAB lfps shape:", loaded["lfps"].shape)

Saved NumPy archive: /home/azosion/Projects/comp-neuro/final code/rec04_lfps.npz
Saved MATLAB file: /home/azosion/Projects/comp-neuro/final code/rec04_lfps.mat
